In [4]:
!pip install pandas
!pip install torch
!pip install transformers
!pip install scikit-learn
!pip install numpy
!pip install urllib3

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
import torch
import numpy as np
import pandas as pd
import urllib.request
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup


# Load your dataset by simply uncomment any one of below
url = "<dataset-url>"  # dataset CSV location removed; set before running

#download and read the contents
headers = {'User-Agent': 'Mozilla/5.0'}
request = urllib.request.Request(url, headers=headers)
df = pd.read_csv(urllib.request.urlopen(request), encoding='utf-16')

# Initialize the tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
model = BertForSequenceClassification.from_pretrained('bert-base-chinese', num_labels=2)
model.pooler = torch.nn.Linear(model.config.hidden_size, 2)  # 2 classes for binary classification

# Tokenize the data
tokenized = df['hex'].apply((lambda x: tokenizer.encode(x, add_special_tokens=True, max_length=512, truncation=True)))

# Pad the data
max_len = 0
for i in tokenized.values:
    if len(i) > max_len:
        max_len = len(i)
padded = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values])
padded.shape

# Create attention masks
attention_mask = np.where(padded != 0, 1, 0)
attention_mask.shape

# Convert to tensors
input_ids = torch.tensor(padded)
attention_mask = torch.tensor(attention_mask)
labels = torch.tensor(df['label'])

# Create the dataset
dataset = TensorDataset(input_ids, attention_mask, labels)

# Split the dataset
train_size = int(0.3 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create the dataloaders
batch_size = 32
train_dataloader = DataLoader(train_dataset, sampler=None, batch_size=batch_size)
val_dataloader = DataLoader(val_dataset, sampler=None, batch_size=batch_size)

# Set the optimizer
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

# Set the number of epochs
epochs = 10
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# Set the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Define the training loop
def train():
    model.train()
    total_train_loss = 0
    for step, batch in enumerate(train_dataloader):
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)
        model.zero_grad()
        outputs = model(batch_input_ids, token_type_ids=None, attention_mask=batch_attention_mask, labels=batch_labels)
        loss = outputs[0]
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    avg_train_loss = total_train_loss / len(train_dataloader)
    return avg_train_loss


# Train the model
for epoch in range(epochs):
    train_loss = train()
    model.eval()
    total_val_loss = 0
    predictions, true_labels = [], []
    for batch in val_dataloader:
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)
        with torch.no_grad():
            outputs = model(batch_input_ids, token_type_ids=None, attention_mask=batch_attention_mask)
        logits = outputs[0]
        logits = logits.detach().cpu().numpy()
        label_ids = batch_labels.to('cpu').numpy()
        predictions.append(logits)
        true_labels.append(label_ids)

    flat_predictions = np.concatenate(predictions, axis=0)
    flat_predictions = np.argmax(flat_predictions, axis=1).flatten()
    flat_true_labels = np.concatenate(true_labels, axis=0)
    val_accuracy = accuracy_score(flat_true_labels, flat_predictions)
    print('Epoch:', epoch+1)
    print('Training loss:', train_loss)
    #print('Validation accuracy:', val_accuracy)

# Define the prediction function
# Define the prediction function
def predict(text):
    encoded = tokenizer.encode_plus(
        text,
        max_length=512,
        truncation=True,
        add_special_tokens=True,
        return_token_type_ids=False,
        return_attention_mask=True,
        return_tensors='pt',
    )
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    output = model(input_ids, attention_mask)
    print(output)
    _, prediction = torch.max(output[0], dim=1)
    return str(prediction.item())


# Test the model, 01111
test  = predict("㌷㔶㐷㘴㤶挶㔶挶㤶㌷㐷㠲戵戵搵搵㤲戳㐶㘶㠶㜶㔶㈷㌷㠶挴昶㈷㔶搶〲㤶〷㌷㔷搶〲㐶昶挶昶㈷〲㌷㤶㐷〲ㄶ搶㔶㐷挲〲㌶昶收㌷㔶㐷㔶㐷㔷㈷〲㌷ㄶ㐶㤶〷㌷㌶㤶收㜶〲㔶挶㤶㐷㈷挲〲㌷㔶㐶〲㐶㤶ㄶ搶〲收昶收㔷搶㤷〲㔶㤶㈷搶昶㐶〲㐷㔶搶〷昶㈷〲㤶收㘷㤶㐶㔷收㐷〲㔷㐷〲挶ㄶ㈶昶㈷㔶〲㔶㐷〲㐶昶挶昶㈷㔶〲搶ㄶ㜶收ㄶ〲ㄶ挶㤶ㄷ㔷㤷ㄶ搶〲㔶㈷ㄶ㐷挲〲㌷㔶㐶〲㐶㤶ㄶ搶〲㘷昶挶㔷〷㐷㔷ㄶ攲〲ㄴ㐷〲㘷㔶㈷昶〲㔶昶㌷〲㔶㐷〲ㄶ㌶㌶㔷㌷ㄶ搶〲㔶㐷〲愶㔷㌷㐷昶〲㐶㔷昶〲㐶昶挶昶㈷㔶㌷〲㔶㐷〲㔶ㄶ〲㈷㔶㈶㔷搶攲〲㌵㐷㔶㐷〲㌶挶㤶㐷ㄶ〲戶ㄶ㌷㐶〲㜶㔷㈶㔶㈷㜶㈷㔶收挲〲收昶〲㌷㔶ㄶ〲㐷ㄶ戶㤶搶ㄶ㐷ㄶ〲㌷ㄶ收㌶㐷㔷㌷〲㔶㌷㐷〲挴昶㈷㔶搶〲㤶〷㌷㔷搶〲㐶昶挶昶㈷〲㌷㤶㐷〲ㄶ搶㔶㐷攲〲挴昶㈷㔶搶〲㤶〷㌷㔷搶〲㐶昶挶昶㈷〲㌷㤶㐷〲ㄶ搶㔶㐷挲〲㌶昶收㌷㔶㐷㔶㐷㔷㈷〲㌷ㄶ㐶㤶〷㌷㌶㤶收㜶〲㔶挶㤶㐷㈷挲〲㌷㔶㐶〲㐶㤶ㄶ搶〲收昶收㔷搶㤷〲㔶㤶㈷搶昶㐶〲㐷㔶搶〷昶㈷〲㤶收㘷㤶㐶㔷收㐷〲㔷㐷〲挶ㄶ㈶昶㈷㔶〲㔶㐷〲㐶昶挶昶㈷㔶〲搶ㄶ㜶收ㄶ〲ㄶ挶㤶ㄷ㔷㤷ㄶ搶〲㔶㈷ㄶ㐷挲〲㌷㔶㐶〲㐶㤶ㄶ搶〲㘷昶挶㔷〷㐷㔷ㄶ攲〲ㄴ㐷〲㘷㔶㈷昶〲㔶昶㌷〲㔶㐷〲ㄶ㌶㌶㔷㌷ㄶ搶〲㔶㐷〲愶㔷㌷㐷昶〲㐶㔷昶〲㐶昶挶昶㈷㔶㌷〲㔶㐷〲㔶ㄶ〲㈷㔶㈶㔷搶攲〲㌵㐷㔶㐷〲㌶挶㤶㐷ㄶ〲戶ㄶ㌷㐶〲㜶㔷㈶㔶㈷㜶㈷㔶收挲〲收昶〲㌷㔶ㄶ〲㐷ㄶ戶㤶搶ㄶ㐷ㄶ〲㌷ㄶ收㌶㐷㔷㌷〲㔶㌷㐷〲挴昶㈷㔶搶〲㤶〷㌷㔷搶〲㐶昶挶昶㈷〲㌷㤶㐷〲ㄶ搶㔶㐷攲")
test1 = predict("慥愳㡣㜱〱㐲ち昴㜳ㅢ晢㐰㉤晥㜵慣捦㌱㔷㤳㘶㜴㘷㘵扣攷ぢ㉡㕥挶㜳㘱㕢扥愴ㅤ㘵㥣搴㑥㠶㔰〴㘲晤挲㥥㍣晥摣㕣愳㑣㡣晤㠷㘴搲㠸㤵晦ㅢ㝣ㅡ攰ㄶ㘰捡愴㤵㙢㑥㍦㉤昸㝣慢㝦㜵慣挹昵戵㕤㐷昰敤㠱㠰㑤㙡㝦ㄲㅡ〰搳昷㤶搲㘰愱挳㤱㘳㜸ㄶ㐹摢昹ㄸ㝥㙣摤㡦敥㉤㔱慣㍦摢㑢搷㔵㍥㜰づ挹愱㘱扤ㅡ敦㤴㑦㉦㝣㐰愱㔲晢て㐶ぢ戴愵㡣愸㝤㡦愶㄰捦㜰㥡ㄸ捡ㄲ搵挷敦㈷㡦户㈸㌷慡㝡㤴㝡扣㠱㘱㘰㙣て改戳㕡㝦㐵扣㍤㉣搰〶扤㉢㘴ㄲ㌵㥤㠶㕢攴㐲㉤㌹㜱㕦ㄳ〷晦㙦搶㘵㙣〴搶㤹挷ㅥ〴愲㔴昳愹㥡捦慥㜷捥戵㌶愹㘶㙤㍡ㄸ愲㘹㍣戰扣㜱㜱㥡㥦㥥戱㜸晥㔲㤵㉥扦〶㈹㔸㘶㠷㌲㑣攷昱㡣搳つ㔹摢扤㌵㜱昸㜷㥤搰っ昰㔶〴晢㍦昰攳愵攰摣㝢㡤㡦㘰㤶愴捥㤸㜴㑣扢㜶慦㈸挰慦㘸㐸㙢㈵慦㘳戲攰㈸挳摢〴㥥㔳〱㌱㠷愲㘹㔰ㄷ㜸ㅥ㥥扦㘶敡愵愸昲㕡㐹敥㍦扦搹㘵㔷㠶㑣㠴晦㠷晡㜰㑡晥㝦〴户㔹㥢摡㐶戹ぢ㝣扡〶㕦ㄵ㌵攷ㅦ㜰㐹昷敢㉦㘵㑦搴挵㤶㤴戵㐶ㅦ㔶㍡㥡搷㈱㜷戵㑡搶戲愶㤳〳㤶㘰㙢戸愷㉡㝥ㄷ㈴ㅣ收㈸㠷ㅡ挵㍢扣㙦攰挳晣㠳攲ㄳ㈲捤㤷㥦㜹㑤㡢〸㔸㕢㍣㘰㜸〶ㄸ㌵㌱攲㥡戴㔳攳㔲捡昴挷晦戶㍣㈱散摡ㄴ㤶㝥摣攳㔱㠴〷㐰昲㘵晤戵晤挳慡攲㠸㌹㥡づ挸㔹㑦敦摤㉦捡㙦㠹愳づ攰㍢㜷㜸㘶戱捣戴㤴㜰㌰㠱昴戶㤳搷㡣㜲〳㔲戶戸愱㔴〲慡愶攳㉣摢敢㜴昷㍡晢挲㝥昰戶㈰")
test2 = predict("㈸扣㔱㕥〳㈱昹㙢搰敤㘴㔹搵㤱㔰昷㌲㐱愱㥡愱㔹㕣挵ㄵㅤ㘰㑤㡢扦㌱愷㥦攵晢挰㍢愲〲散〶愸㌳晤〵㝥㉦ㄶ〷ㅡ㝣㘲㝡㐷搳㘱㠶㌷㤸愵㈴㙡㕣㠳ㅥ㉥昷戹㐲㤹扥ㅡㄶ愱攷ㄸ㌹㝦戵愷愱戳㤹㜴㌳㐹㝥㜶戰〹㈰㈴慢㥢昴㤳摥㘸㜳搷户扦㈱捡挶㜵㠱晣㉥㔶㈲㘵㈲㜹㝥㘵㈳挲㙡〵㘳扦㤱㘹戵捥㜹㜰捡攰㜹㕥戳愱昸㘱ㄵ㤶扦㔲〴㐵㐵搱㜷敤〵〱㌳昷愸㉥愷㠹挸㠷搷㄰㜲戹㘷㕡㕦㈲㠳㑤㠵搳扥捣㡥㈳㈶㑢攸搶㉣挴挰摤て攳㘲㘳扦挳㈲㙣㐰㐲捡つ昶改搱㠲㙤敦ㅢ㡣扥㤶㝦㈲扥㐹㠳昳搳挴〰慣㝣㍡づ攸て㠲㜷㌰挷搲挰㤰㜲ㄲ愰㜸搶摣㉦扥㤳挴㤴㡤ㄶ㐹搱㙣敡㔷㔳㑦挰ㅦ扥㈵㉣㜸挳㔲㡦挰㑥挶〳㉢戱ㄹ昱㐹挱㌷戲㤰扥慥㍦㌳㜴ㅤ㘲挰㡣愶㌲㜴ㅦ㝦〷㥤慡攸㜰搰㤳㠹戰㍡㙤敥搳慣㝥㘸㉡㥦敦㍢㌲敡㉤㐵ㄴㄹ愳散㡤敥㤹㌱㐰慣扦㍥㕣㈰㘶搲㌱ㄱ〹㘰㤰挱挰搵搲㕤扦㙣攳㘶〳ㅥ晡ㄸ㘴戶㐰敦搲扦攵㠵㘷挴敦㑡㈵㍢搶㉦㜶㉦戸敥戲ㄳ㜹昰戱ㅥ㤰扡㝦㔹㉤挸敡敤㠵晡慦昴挷㈳㌲㤴扤挹捥愷戲ㄵ戲㥤挹摦㔱搷㔱ㄴ㉦㐶㙣昱戲〶㐳扦挱㠰㈳晥攴㝣摣つ攱㜶ㅦ晦敡摡〳㐰扦㤰〷㝥挶㤷㌳㤵㈴㈳㙣愲搴慢㠲㠰㌱㈹㕥昶㈱㜷慢㤱摦晤㙢挹㜶㍡敥㠴㐹㉣愱ㄷ搰ㄶ户㕥昴㠸㐷㔸昲㌴㙤㠵㄰㥦㐱〹㌹㐳㡦〵〵㠷攰㤶㤲ㅢ㠱昵攰㉣㍤㘹〶捤攳㍡捤挵ㄳ㌹昹ㄲ㑣挶戸㝥㑢㘱挴ち愲㡤收戶㉦㕣㠴㠷")
test3 = predict("挵㠱㍡㜲㠷㕦敦㘱㘵㠹昱ㄷ㑡㤸㌹晣㠹晢搶捥愹攸㜰昰扦㔱〸㠴㐳戴昵敦㙦㈰㑤搵搲戸戵㥡ㅥ㤷〶慦㜹ㄱ戸晥〶㤳㝣㜰㍣〴㤰㔷㔳㡣ㄲ攰㕡㜲㘸㠷㔳㤵摦㌱㔶摤戸㑥挱戸扥㉡つ戵㙡㕤捣挶〵〱戸㑡㜶搱㈳〷ㅥ㠷〴戰攵㕡㌳摤㍥挱㘷㍤㠳昳ㄸ㕢㜳戰㠷攵㥢㉣晡㈱捡㜸昶㘲挱户慡㤰㙤敥㙢㑡㤳㠰昲㉦搹㘴敥㜲愳ㅤづ㘵ㄷ摥昸㡢㍥慢㈱㝥〱㌲愴搹㐴㐷戸㔴昶搶搰换㌳ㄵ㍡㍡攴㍥㍢愰戱摤㌹㤳挴户搳戶㥤愲搱㕤㑥摤㠴㐳㙤㈸戵昴㘲㈸㙤㕢ㄸ㝦㈰慡㉥愴〰昲愴晣敡㙥搷晤㝢㐵㍢㔰〴㈴搶㙥㝦㡣挴㙤㡦㙥㠴改扣戶昱㘴晡㜰户㜸捤㤵㝡㙤㔰㈴捡㤶㜲㑡㘶㕦〸ㄷ㐳㉦づ㌵㥥户㥣㕢㑡扦㙥㌵晤㔹㤰搴㐷晦つ慥㙥㠷㐰㤶㜰㘶ㅢ㤸㙢㠱〳㘳㜶㕤搹〴㥣摦㝣攳㑣愸㍣㕢ㄹ捥挸㐶㑤摦㥣晢㡥挰挱攸改㙤㈴㜰㔰昴敥㥢㝥昰㜶㐰愲昹㤰戹㈳㤵㝢愱ㄷ㐵㘶扢㘶摡㤲㈴慥改㝥攵搲㔴㍣扥搲扢㉣㌱挵㡢挹㡢昹扢敥㥢挰㙣搹攳㔵㌳㈵㙤搹㈸㄰慦慣㘶挲㤶搳㜰㠶㡣つ攳㉢扣㕦㡤挵㜳搷搲晡戵慣㠳散㝦ㄱ㍥㘸㥦戰㘳㡦㤵〲㜳㠵㤳攴㘰搰㜱㝤㤷㐷㈶晦ㄹ敡捦㉢昴敥搳ㄹ㝢㠷挹㐶ぢ㉣攴㍢ㄳ愵捤换㉢攰㠷㝦晣㌶㜵搰戵搸敡㄰㘸㑤㍣ㅥ搹ち〶昰捦捣愰攱换㘱摤㉥㍡扥㈶摢㉢扥〱㕥〹㝥慦〷晤慡捦㑦㌶挶愵愶㡣愶ㅢ慦ㄴ昰㌲㤳慤摤㔵㑥愴㍣愱㠵㠷㐴ㄵ摢㠷摥搹㔵㔷昳搹捦㍦て㍢㠰ㄱ攸㝢愷㡦㠰㙣㜹㈲")
test4 = predict("搴愵〹〰㌰〰〰〰㐰〰〰〰晦晦〰〰㡢〰〰〰〰〰〰〰〴〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰て〰〰〰攰昱慢攰〰㑢㤰摣ㄲ㡢㄰挴摣ㄲ㐵㠶㤶㌷〲〷㈷昶㜶㈷ㄶ搶〲㌶ㄶ收收昶㐷〲㈶㔶〲㈷㔷收〲㤶收〲㐴昴㌵〲搶昶㐶㔶攲搰搰愰㐲〰〰〰〰〰〰〰愰愲收搶攴戴〰攳攴戴〰攳攴戴〰攳㜴㌳㌹攳㈴戴〰攳搵搲㄰昳挴戴〰攳搵搲㌰昳挴戴〰攳搵搲㔰昳昵戴〰攳搵搲㐰昳㔴戴〰攳挶戲㄰昳㤴戴〰攳攴戴㄰攳〷戴〰攳ㄷ挲㤰昳昴戴〰攳ㄷ挲晦攳昴戴〰攳ㄷ挲㈰昳昴戴〰攳㈵㤶㌶㠶攴戴〰攳〰〰〰〰〰〰〰〰〵㔴〰〰㐶㘸㘰〰攸㐶挵戵〰〰〰〰〰〰〰〰て〰㈲〰戰㈰攰攰〰㈱〰〰〰〲〰〰〰〰〰〰㐰㘱〰〰〰〱〰〰〰〰〰〴㄰〰〰〰〰〱〰〰〰㈰〰〰㘰〰〰〰〰〰〰〰㘰〰〰〰〰〰〰〰〰〹〰〰〰㐰〰〰〰〰〰〰㈰〰〶ㄸ〰〰〱〰〰〰〰〰〰〱〰〰〰〰〰〰〰〰〱〰〰〰〰〰〰〱〰〰〰〰〰〰〰〰〰〰〱〰〰〰〰〰〰〰〰〰〰〰㐹㠳〰〰捤〰〰〰〰〷〰〰づ㄰〰〰〰〶〰〰㑡㄰〰〰〰〰〰〰〰〰〰〰〰〸〰〰挱〰〰〰づ㈳〰〰〷〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〵㌳〰〰〰㄰〰〰〰〰〰〰〰〰〰〰〰〳〰〰〲㈰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰〰攲㐷㔶㠷㐷〰〰〰㠳ㄱ〰")

print(test + "" + test1 + "" + test2 + "" + test3 + "" + test4 )

# Save the model in pt format
with open('/content/drive/My Drive/identifibert2.pt', 'wb') as f:
  torch.save(model, f)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch: 1
Training loss: 0.6017181277275085
Epoch: 2
Training loss: 0.5065733194351196
Epoch: 3
Training loss: 0.3384658098220825
Epoch: 4
Training loss: 0.34281179308891296
Epoch: 5
Training loss: 0.2736426591873169
Epoch: 6
Training loss: 0.23559045791625977
Epoch: 7
Training loss: 0.21621857583522797
Epoch: 8
Training loss: 0.17179220914840698
Epoch: 9
Training loss: 0.1766449213027954
Epoch: 10
Training loss: 0.13165801763534546
SequenceClassifierOutput(loss=None, logits=tensor([[ 1.3640, -1.0567]], device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
SequenceClassifierOutput(loss=None, logits=tensor([[-0.3979,  1.0689]], device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
SequenceClassifierOutput(loss=None, logits=tensor([[-0.4374,  0.9891]], device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
SequenceClassifierOutput(loss=None, logits=tensor([[-0.3463,  1.0436]], device='cuda:0', grad_fn=<AddmmBa